In [1]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 25.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 35.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 106.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 104.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 83.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.9 MB/s eta 0:00:00
 

In [2]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import DPOTrainer, DPOConfig
from unsloth.chat_templates import get_chat_template

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
max_seq_length = 2048 
dtype = None          
load_in_4bit = True   

In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/kaggle/input/models/hridaypatel75/oncology-sft/transformers/default/1/kaggle/working/qwen2.5-3b-oncology-chat", 
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.6.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/Qwen2.5-3B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [5]:
for name, param in model.named_parameters():
    if "lora" in name:
        param.requires_grad = True

In [6]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", 
)

Unsloth: Will map <|im_end|> to EOS = <|endoftext|>.


In [7]:
from huggingface_hub import login
login()

In [8]:
dataset = load_dataset("ai-galileo/clinical-notes-to-fhir", split="train")

README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

balanced_train_set.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/530 [00:00<?, ? examples/s]

In [9]:
dpo_dataset = dataset.filter(lambda x: x.get("valid") is False)

Filter:   0%|          | 0/530 [00:00<?, ? examples/s]

In [11]:
def format_dpo_pairs(example):

    user_prompt = str(example.get("note", ""))
    
    good_answer = "The clinical note has been reviewed and properly formatted without structural errors."
    bad_answer = "Error: " + str(example.get("validation_errors", "Invalid formatting detected."))
    
    prompt_chat = [
        {"role": "system", "content": "You are a helpful, empathetic, and expert medical professional."},
        {"role": "user", "content": user_prompt}
    ]
    
    formatted_prompt = tokenizer.apply_chat_template(prompt_chat, tokenize=False, add_generation_prompt=True)
    
    return {
        "prompt": formatted_prompt,
        "chosen": good_answer + tokenizer.eos_token,   
        "rejected": bad_answer + tokenizer.eos_token   
    }

In [12]:
original_columns = dpo_dataset.column_names
dpo_dataset = dpo_dataset.map(format_dpo_pairs, remove_columns=original_columns)

Map:   0%|          | 0/223 [00:00<?, ? examples/s]

In [13]:
dpo_dataset

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 223
})

In [14]:
dpo_dataset[0]

{'prompt': '<|im_start|>system\nYou are a helpful, empathetic, and expert medical professional.<|im_end|>\n<|im_start|>user\nMrs. Alice Smith, a 38-year-old female, presented to the Emergency Department on October 26, 2023, following a fall at home, resulting in a suspected right ankle injury. Emergency medical transport was provided by County EMS. Upon arrival, an X-ray confirmed a fracture of the talus bone in the right ankle. A short leg cast was applied to immobilize the injury. A routine blood specimen was collected for pre-operative lab work. Supplies for the cast were delivered and utilized. A comprehensive claim for all services rendered, including ambulance, hospital care, imaging, casting, and labs, will be submitted to her insurer.<|im_end|>\n<|im_start|>assistant\n',
 'chosen': 'The clinical note has been reviewed and properly formatted without structural errors.<|im_end|>',
 'rejected': 'Error: ["Claim/claim-10001.item[1]: unknown property \\"servicedDateTime\\"", "Claim/c

In [15]:
args = DPOConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60, 
    learning_rate = 5e-6,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs_dpo",
    beta = 0.1,
)

trainer = DPOTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dpo_dataset,
    args = args,
)

Extracting prompt in train dataset (num_proc=8):   0%|          | 0/223 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=8):   0%|          | 0/223 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=8):   0%|          | 0/223 [00:00<?, ? examples/s]

In [16]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 223 | Num Epochs = 3 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
1,0.676499,1.543212,1.500888,0.500000,0.042324,-55.737331,-149.453674,-0.155789,-0.255442
2,0.668308,1.475303,1.392200,0.625000,0.083103,-56.370506,-128.065186,-0.096015,-0.380500
3,0.704621,1.444627,1.445553,0.500000,-0.000926,-56.391922,-127.997704,-0.104632,-0.274203
4,0.641658,1.524461,1.412622,0.625000,0.111840,-54.873871,-114.661118,-0.113634,-0.089981
5,0.638443,1.624846,1.506164,0.500000,0.118682,-54.007099,-118.367538,-0.055786,-0.162956
6,0.536366,1.643373,1.294499,1.000000,0.348873,-52.821922,-121.002739,-0.187694,-0.090619
7,0.496371,1.731076,1.281811,1.000000,0.449265,-53.372520,-160.560867,-0.197091,-0.424113
8,0.500834,1.591758,1.141456,1.000000,0.450302,-54.702080,-133.183136,-0.046782,-0.241971
9,0.381144,1.848968,1.067291,1.000000,0.781677,-53.063210,-145.443970,-0.188361,-0.440874
10,0.371964,1.867251,1.063417,1.000000,0.803834,-51.825935,-141.504440,-0.187614,-0.233011


Unsloth: Restored added_tokens_decoder metadata in outputs_dpo/checkpoint-60/tokenizer_config.json.


In [17]:
final_dpo_name = "qwen25-3b-oncology-dpo-aligned"
model.save_pretrained(final_dpo_name)
tokenizer.save_pretrained(final_dpo_name)

Unsloth: Restored added_tokens_decoder metadata in qwen25-3b-oncology-dpo-aligned/tokenizer_config.json.


('qwen25-3b-oncology-dpo-aligned/tokenizer_config.json',
 'qwen25-3b-oncology-dpo-aligned/chat_template.jinja',
 'qwen25-3b-oncology-dpo-aligned/tokenizer.json')

In [18]:
!zip -r qwen25-3b-oncology-dpo-aligned.zip /kaggle/working/qwen25-3b-oncology-dpo-aligned

  adding: kaggle/working/qwen25-3b-oncology-dpo-aligned/ (stored 0%)
  adding: kaggle/working/qwen25-3b-oncology-dpo-aligned/tokenizer.json (deflated 81%)
  adding: kaggle/working/qwen25-3b-oncology-dpo-aligned/tokenizer_config.json (deflated 90%)
  adding: kaggle/working/qwen25-3b-oncology-dpo-aligned/adapter_config.json (deflated 57%)
  adding: kaggle/working/qwen25-3b-oncology-dpo-aligned/chat_template.jinja (deflated 59%)
  adding: kaggle/working/qwen25-3b-oncology-dpo-aligned/README.md (deflated 65%)
  adding: kaggle/working/qwen25-3b-oncology-dpo-aligned/adapter_model.safetensors (deflated 8%)
